# Comparison between optimal Bayesian and heuristic adversaries

This notebook illustrates a heuristic adversary who assumes that the data they observed is the original data, and uses this assumption to perform linkage attacks, given some auxiliary knowledge about the record of a target who they know is in the dataset.

In [1]:
import itertools
from functools import partial

import numpy as np
import polars as pl
import scipy.sparse as sp
from tqdm.notebook import tqdm

from concurrent.futures import as_completed
from loky import ProcessPoolExecutor
from multiprocessing import get_context

from qif_micro import measure, mechanism, model
from qif_micro import qif
from qif_micro.qif.datatypes import Channel, Joint, ProbabDist, Strategy
from qif_micro.model._internal import _mk_records

## Dynamic perspective - our toy example

Let us first construct the mechanism to be applied to each record:

In [2]:
domain_grade = ["A", "B", "C"]
domain_disability = ["no", "yes"]

# We only need records of length one for this example, but if necessary could
# compute the cross product of the subdomain of records of length 1 and of length m
# to construct the subdomain of records of length m + 1, and then
# concatenate all of subdomains (sounds expensive, though).
domain_records = [
    [{"grade": grade, "disability": dis}]
    for grade in domain_grade
    for dis in domain_disability
]

domain_records

[[{'grade': 'A', 'disability': 'no'}],
 [{'grade': 'A', 'disability': 'yes'}],
 [{'grade': 'B', 'disability': 'no'}],
 [{'grade': 'B', 'disability': 'yes'}],
 [{'grade': 'C', 'disability': 'no'}],
 [{'grade': 'C', 'disability': 'yes'}]]

In [3]:
x = pl.DataFrame({
    "owner_id":   [0,     1,    2,     3,     4],
    "entry_id":   [0,     0,    0,     0,     0],
    "grade":      ["A",   "B",  "C",   "C",   "A"],
    "disability": ["yes", "no", "yes", "yes", "no"]
})

z = pl.DataFrame({
    "owner_id":   [0,     1,    2,     3,     4],
    "entry_id":   [0,     0,    0,     0,     0],
    "grade":      ["A",   "C",  "B",   "C",   "A"],
    "disability": ["yes", "no", "yes", "yes", "no"]
})

Assuming a uniform prior knowledge on records, we get that the optimal adversary would either

- Learn that the target's grade is ``A`` and guess that the disability status is ``yes``;
- Learn that the target's grade is ``B`` and guess that the disability status is ``yes``; or
- Learn that the target's grade is ``C`` and guess that the disability status is ``yes``.

That is, the adversary would guess ``yes`` in all three scenarios.

(Notice that in this notebook we are modelling as if the goal of the adversary was to infer the entire record, but since the adversary learns the student's grade, guessing the entire record is equivalent to inferring the disability status.)

In [4]:
domain_size = len(domain_records)
pi = ProbabDist(np.repeat(1 / domain_size, domain_size))

p = 3/4
eps = np.log(p * (len(domain_grade) - 1) / (1 - p)) # random response with p = 3/4
m_grade = partial(qif.dp.random_response, eps, domain_size=domain_size)
m_record = partial(mechanism.record, grade=m_grade)

baseline_joint, optimal_st = model.generic(
    pi, 
    domain_records, 
    m_record, 
    baseline_dataset=x, 
    sanitised_dataset=z, 
    hint="grade"
)

optimal_st.dist.toarray()

array([[0., 0., 0.],
       [1., 0., 0.],
       [0., 0., 0.],
       [0., 1., 0.],
       [0., 0., 0.],
       [0., 0., 1.]])

The adversary's chance of correctly inferring the target's disability status is thus $60\%$:

In [5]:
measure.linkage_risk((baseline_joint, optimal_st))

np.float64(0.6000000000000001)

Now, the heuristic works as follows: the adversary observes the dataset $z$, but assumes this is the real data, and so constructs their strategy as if they had observed the result of a pipeline without any post-processing. There is a chance that, by following this heuristic, the adversary doesn't see some of the possible hints as possible, and so they do not further update their knowledge in these scenarios.

In [6]:
def heuristic_model(
    x: pl.DataFrame,
    z: pl.DataFrame,
    hint: str
) -> tuple[Channel, Strategy]:
    id_cols = ["owner_id", "entry_id"]
    attrs = [c for c in x.collect_schema().keys() if c not in id_cols]
    entry_expr = pl.col("entry_id").cast(pl.UInt64)

    # Fix order of columns, to avoid some non-determinism
    x = x.select("owner_id", entry_expr, *attrs)
    z = z.select("owner_id", entry_expr, *attrs)

    baseline_model = model.baseline(
        x, hint, return_owners=True, return_labels=True
    )
    
    heuristic_model = model.baseline(
        z, hint, return_owners=True, return_labels=True
    )

    # We start by reindexing the baseline joint. We need to reindex both rows and columns.
    # For rows, we find the records that are common in the baseline and heuristic, and we
    # reindex so that these records como first in both model. Then, we complete the baseline
    # joint with the remaining rows, followed by the rows that correspond to records that
    # are possible in the heuristic model, but not in the baseline model.
    #
    # For columns, we follow a similar approach, but we only care about columns present
    # in the baseline: we move common columns to the beginning, and keep the rest at the end.
    baseline_records = (
        _mk_records(x).lazy().unique("record")
        .join(baseline_model[1], on="owner_id")
        .drop("owner_id")
        .rename({"record_right": "row_baseline"})
    )

    heuristic_records = (
        _mk_records(z).lazy().unique("record")
        .join(heuristic_model[1], on="owner_id")
        .drop("owner_id")
        .rename({"record_right": "row_heuristic"})
    )

    common_rows = (
        baseline_records
        .join(heuristic_records, on="record")
        .drop("record")
        .collect().lazy()
    )

    missing_rows_heuristic = (
        baseline_records
        .join(heuristic_records, on="record", how="anti")
        .drop("record")
    )

    missing_rows_baseline = (
        heuristic_records
        .join(baseline_records, on="record", how="anti")
        .drop("record")
        .with_row_index("row_baseline", offset=baseline_model[0].dist.shape[0])
    )

    common_cols = (
        baseline_model[2]
        .join(heuristic_model[2], on="hint_label")
        .drop("hint_label")
        .rename({"hint_right": "hint_heuristic"})
        .collect().lazy()
    )

    reindex_rows = pl.concat([
        common_rows.select("row_baseline"),
        missing_rows_heuristic.select("row_baseline"),
    ]).collect().to_numpy().ravel()

    reindex_cols = pl.concat([
        # Start with the cols that are common to both baseline and heuristic:
        common_cols.select("hint"),
        # Then get the remaining indices and keep them at the end.
        # The order of the remaining indices does not really matter
        baseline_model[2].join(common_cols, on="hint", how="anti").select("hint")
    ]).collect().to_numpy().ravel()

    n_cols = reindex_cols.shape[0]
    n_rows = reindex_rows.shape[0]
    n_extend_rows = missing_rows_baseline.select(pl.len()).collect().item()
    
    dist = baseline_model[0].dist[reindex_rows, :]
    dist = dist[:, reindex_cols]
    
    missing_rows = sp.csr_array((n_extend_rows, n_cols), dtype=np.float64)
    baseline_joint = Joint(sp.vstack([dist, missing_rows]))

    # Now, for the heuristic adversary, we match the same format of the baseline joint:
    # a. We reindex rows so that the first rows match with the records present in both datasets,
    #    followed by rows that correspond to records present only in the baseline dataset,
    #    and finally rows that correspond to records present only in the sanitised dataset.
    #
    # b. Then we reindex columns, ignoring columns that are not present in the baseline.
    #    Again, we match with the columns in the baseline joint: first the columns common
    #    to both models, and then the columns present only in the baseline.
    # 
    # For columns in the baseline not present in the heuristic model, we assume that
    # the adversary will fallback to their intermediate knowledge, upon observing z.
    # 
    # After reindexing, we then redistribute weights in the joint so that it adds up to 1,
    # noting that this will not affect in any way the construction of strategies.
    common_cols = common_cols.select("hint_heuristic").collect().to_numpy().ravel()
    common_rows = common_rows.select("row_heuristic").collect().to_numpy().ravel()
    missing_rows_baseline = missing_rows_baseline.select("row_heuristic").collect().to_numpy().ravel()
    
    delta = heuristic_model[0].dist.sum(axis=1)
    
    n_extend_rows = missing_rows_heuristic.select(pl.len()).collect().item()
    missing_rows = sp.csr_array((n_extend_rows, n_cols), dtype=np.float64)

    n_rows = heuristic_model[0].dist.shape[0]
    n_extend_cols = n_cols - common_cols.shape[0]
    missing_cols = sp.lil_array((n_rows, n_extend_cols), dtype=np.float64)
    missing_cols[:, :] = delta[:, np.newaxis]
    missing_cols = missing_cols.tocsc()

    dist = heuristic_model[0].dist.tocsc()
    dist = sp.hstack([dist[:, common_cols], missing_cols])
    dist = sp.vstack([dist[common_rows, :], missing_rows, dist[missing_rows_baseline, :]])
    dist = dist / dist.sum()

    adv_st = qif.strategy(Joint(dist.tocsr()))
    return baseline_joint, adv_st

In [7]:
baseline_joint, adv_st = heuristic_model(x, z, "grade")
baseline_joint.dist.toarray()

array([[0. , 0.2, 0. ],
       [0.4, 0. , 0. ],
       [0. , 0.2, 0. ],
       [0. , 0. , 0.2],
       [0. , 0. , 0. ],
       [0. , 0. , 0. ]])

By following the heuristic in this particular scenario, the adversary would

- Learn that the target's grade is ``A`` and guess that the disability status at random;
- Learn that the target's grade is ``B`` and guess that the disability status is ``yes``; or
- Learn that the target's grade is ``C`` and guess that the disability status at random.
  
This means that the heuristic adversary would perform worse, on average (over all possible grades), than the optimal Bayesian adversary:

In [8]:
measure.linkage_risk((baseline_joint, adv_st))

np.float64(0.4)

Naturally, there are scenarios in which the heuristic adversary performs better than the optimal

In [9]:
x = pl.DataFrame({
    "owner_id":   [0,     1,    2,     3,     4],
    "entry_id":   [0,     0,    0,     0,     0],
    "grade":      ["A",   "A",  "B",   "B",   "A"],
    "disability": ["yes", "no", "yes", "yes", "no"]
})

baseline_joint, optimal_st = model.generic(
    pi, 
    domain_records, 
    m_record, 
    baseline_dataset=x,
    sanitised_dataset=z, 
    hint="grade"
)

optimal_risk = measure.linkage_risk((baseline_joint, optimal_st))

baseline_joint, heuristic_st = heuristic_model(x, z, "grade")
heuristic_risk = measure.linkage_risk((baseline_joint, heuristic_st))

optimal_risk, heuristic_risk

(np.float64(0.6000000000000001), np.float64(0.7))

Nevertheless, on average over all datasets that could have been remapped to the sanitised dataset $z$ above, the optimal adversary still peforms better:

In [10]:
def eval_fixed_output(z, n_records, n_datasets, domain_grade, p_keep_grade):
    slice_grade = list(itertools.product(domain_grade, repeat=5))
    
    p_repl_grade = (1 - p_keep_grade) / (len(domain_grade) - 1)
    
    p_expr = (
        pl.when(pl.col("grade") == pl.col("grade_right"))
        .then(p_keep_grade)
        .otherwise(p_repl_grade)
        .alias("p")
    )

    p_z = 0
    optimal_risk = 0
    heuristic_risk = 0

    disability = z["disability"].to_list()

    for baseline_grade in tqdm(slice_grade):
        x = {"grade": baseline_grade, "disability": disability}
        
        x = (
            pl.DataFrame(x)
            .with_row_index("owner_id")
            .with_columns(pl.row_index("entry_id").over("owner_id"))
        )
        
        weight = (
            x.join(z, on="owner_id")
            .with_columns(p_expr)
            .group_by("owner_id")
            .agg(pl.col("p").product())
            .select("p").product().item()
        ) / n_datasets
    
        p_z += weight
    
        adv_model = model.generic(
            pi, 
            domain_records, 
            m_record, 
            baseline_dataset=x,
            sanitised_dataset=z,
            hint="grade"
        )
        
        curr_optimal_risk = measure.linkage_risk(adv_model)
        optimal_risk += weight * curr_optimal_risk
        
        adv_model = heuristic_model(x, z, "grade")
        curr_heuristic_risk = measure.linkage_risk(adv_model)
        heuristic_risk += weight * curr_heuristic_risk

    return optimal_risk, heuristic_risk, p_z

In [11]:
n_records = len(domain_records)
n_datasets = n_records**5

optimal_risk, heuristic_risk, p_z = eval_fixed_output(
    z, n_records, n_datasets, domain_grade, p_keep_grade=p
)

# Divide by the probability of observing z, given that is has already been observed:
optimal_risk /= p_z
heuristic_risk /= p_z

  0%|          | 0/243 [00:00<?, ?it/s]

In [12]:
optimal_risk, heuristic_risk

(np.float64(0.5999999999999973), np.float64(0.5749999999999974))

## Static perspective

We already now empirically that the heuristic is not optimal for a fixed sanitised dataset, on average over all possible input datasets.
We now show that it is also not optimal under a static perspective.

In [13]:
def eval_fixed_disability(disability, n_records, n_datasets, domain_grade, p_keep_grade):
    optimal_risk = 0
    heuristic_risk = 0

    slice_grade = itertools.product(domain_grade, repeat=5)
    for sanitised_grade in slice_grade:        
        z = (
            pl.DataFrame({"grade": sanitised_grade, "disability": disability})
            .with_row_index("owner_id")
            .with_columns(pl.row_index("entry_id").over("owner_id"))
        )


        result = eval_fixed_output(
            z, n_records, n_datasets, domain_grade, p_keep_grade
        )
        
        optimal_risk += result[0]
        heuristic_risk += result[1]

    return optimal_risk, heuristic_risk

In [ ]:
slice_disability = list(itertools.product(domain_disability, repeat=5))

optimal_risk = 0
heuristic_risk = 0

ctx = get_context("spawn")
with (
    ProcessPoolExecutor(max_workers=4, context=ctx) as pool,
    tqdm(total=len(slice_disability)) as pbar
):
    futures = (
        pool.submit(eval_fixed_disability, d, n_records, n_datasets, domain_grade, p)
        for d in slice_disability
    )

    for future in as_completed(futures):
        result = future.result()
        optimal_risk += result[0]
        heuristic_risk += result[1]

        pbar.update(1)
        pbar.set_postfix({
            "optimal": optimal_risk, 
            "heuristic": heuristic_risk,
            "diff": optimal_risk - heuristic_risk
        })

  0%|          | 0/32 [00:00<?, ?it/s]